# Imports

In [53]:
import os
import json
import joblib
from pathlib import Path

In [2]:
import numpy as np
import pandas as pd

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from sklearn.linear_model import LinearRegression

In [5]:
from train_model import TrainModel

In [6]:
data_store=Path("./Prepared_Data_Store")

# Importing Data

In [7]:
data=pd.read_csv(os.path.join(data_store,"stocks_data.csv"),
                 parse_dates=["date"],
                 index_col=["date","ticker"])

In [8]:
data

open        high         low       close     volume  \
date       ticker                                                              
2010-02-22 A        21.347311   21.347311   21.047125   21.251797  2888500.0   
           ACGL     24.070000   24.306667   24.070000   24.230000  2048400.0   
           ACN      34.573465   34.657811   34.202342   34.328862  2421400.0   
           ADI      24.188871   24.261559   23.898120   24.140412  4298200.0   
           ADM      24.683991   24.866038   24.535043   24.667441  3619800.0   
...                       ...         ...         ...         ...        ...   
2017-11-29 XEL      51.180000   51.480000   50.850000   51.260000  2311537.0   
           XOM      81.650000   82.310000   81.480000   82.270000  9475070.0   
           YUM      81.290000   82.120000   81.200000   81.810000  1658544.0   
           ZBH     115.500000  116.740000  115.000000  116.520000  1749712.0   
           ZBRA    111.630000  112.390000  108.575000  109.430000   318334.0   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2010-02-22 A        6.138582e+07      8.755057e+07       8.758384e+07   
           ACGL     4.963273e+07      4.717724e+07       4.194648e+07   
           ACN      8.312391e+07      1.116830e+08       1.292910e+08   
           ADI      1.037603e+08      1.565638e+08       1.256450e+08   
           ADM      8.929120e+07      1.197275e+08       1.496057e+08   
...                          ...               ...                ...   
2017-11-29 XEL      1.184894e+08      1.072401e+08       1.381043e+08   
           XOM      7.795140e+08      6.577386e+08       7.044767e+08   
           YUM      1.356855e+08      1.058911e+08       1.232016e+08   
           ZBH      2.038764e+08      1.181175e+08       1.283747e+08   
           ZBRA     3.483529e+07      2.512778e+07       3.858277e+07   

                   dollar_volume_21d  dollar_volume_rank  ...  bb_upper  \
date       ticker                                         ...             
2010-02-22 A            7.986180e+07               197.0  ...  0.003026   
           ACGL         3.907178e+07               268.0  ...  0.008969   
           ACN          1.358379e+08               119.0  ...  0.033302   
           ADI          1.216604e+08               133.0  ... -0.005210   
           ADM          1.499746e+08               108.0  ...  0.035304   
...                              ...                 ...  ...       ...   
2017-11-29 XEL          1.471875e+08               187.0  ...  0.012771   
           XOM          6.980541e+08                19.0  ...  0.028160   
           YUM          1.427005e+08               192.0  ...  0.009228   
           ZBH          1.856724e+08               151.0  ...  0.015251   
           ZBRA         3.879641e+07               357.0  ...  0.058111   

                   avg_true_range      macd  macd_hist  macd_signal  \
date       ticker                                                     
2010-02-22 A             0.473915  0.056859   0.260081    -0.203223   
           ACGL          0.346561  0.061855   0.111841    -0.049985   
           ACN           0.674139 -0.479265   0.174337    -0.653602   
           ADI           0.650486 -0.198539   0.494893    -0.693432   
           ADM           0.516744 -0.184526   0.023590    -0.208116   
...                           ...       ...        ...          ...   
2017-11-29 XEL           0.538257  0.529041  -0.011455     0.540496   
           XOM           0.746925 -0.283034  -0.052811    -0.230222   
           YUM           1.033082  1.068747   0.145776     0.922972   
           ZBH           2.028270 -0.638798   0.623590    -1.262388   
           ZBRA          2.555431 -0.258970   0.158463    -0.417433   

                   forward_returns_1d  forward_returns_5d  forward_returns_7d  \
date       ticker                                    

# Importing Various Feature Names

## Common Irrelevant Features

In [9]:
with open(os.path.join(data_store,"irrelevant_common_features.json"),"r") as file:
    common_irrelevant_features=json.load(file)

In [10]:
common_irrelevant_features

['returns_15d_lag5', 'returns_21d_lag3', 'momentum_21d_lag3']

## Irrelevant Features Target-1D

In [11]:
with open(os.path.join(data_store,"irrelevant_features_1d.json"),"r") as file:
    irrelevant_features=json.load(file)

In [12]:
irrelevant_features

['dollar_volume',
 'dollar_volume_7d',
 'dollar_volume_15d',
 'returns_7d',
 'returns_15d',
 'returns_1d_lag5',
 'returns_7d_lag5',
 'returns_7d_lag7',
 'returns_15d_lag1',
 'returns_15d_lag3',
 'returns_15d_lag5',
 'returns_15d_lag7',
 'returns_21d_lag3',
 'returns_21d_lag7',
 'momentum_15d',
 'momentum_7d_lag5',
 'momentum_7d_lag7',
 'momentum_15d_lag1',
 'momentum_15d_lag3',
 'momentum_15d_lag5',
 'momentum_21d_lag3',
 'momentum_21d_lag5',
 'momentum_21d_lag7',
 'volatility_7d',
 'volatility_7d_lag1',
 'volatility_7d_lag3',
 'volatility_15d_lag3',
 'volatility_21d_lag7',
 'sma_15d',
 'sma_21d',
 'min_price_21d',
 'max_price_15d',
 'max_price_21d']

## Non-Stationary Features

In [13]:
with open(os.path.join(data_store,"non_stationary_features.json"),"r") as file:
    non_stationary_features=json.load(file)

In [14]:
non_stationary_features

['open',
 'high',
 'low',
 'close',
 'sma_7d',
 'sma_15d',
 'sma_21d',
 'min_price_7d',
 'min_price_15d',
 'min_price_21d',
 'max_price_7d',
 'max_price_15d',
 'max_price_21d',
 'avg_true_range']

# Model Comparison

In [15]:
model_comparison=pd.read_csv(
    os.path.join(data_store,"model_comparison.csv"),
    index_col=["Models"]
)

In [16]:
model_comparison

,Model Description,Cond_No,R-Squared,Log-Likelihood,AIC,BIC,Durbin-Watson,Cond_No_rank,AIC_rank,BIC_rank,R-Squared_rank,Log-Likelihood_rank,Durbin-Watson_rank,Model_overall_rank
Models,,,,,,,,,,,,,,
Model_1,Using all features,841,0.009,1955100,-3910000,-3909000,1.332,9.0,1.0,1.5,1.5,1.0,2.0,1.0
Model_2,Removing common irrelevant features,731,0.007,1954700,-3909000,-3908000,1.330,7.0,2.5,3.5,4.0,2.5,4.0,3.0
Model_3,Removing irrelevant features,357,0.006,1954200,-3908000,-3908000,1.327,3.0,5.0,3.5,5.5,4.0,6.0,4.0
Model_4,Removing non-stationary features,835,0.008,1954700,-3909000,-3909000,1.331,8.0,2.5,1.5,3.0,2.5,3.0,2.0
Model_5,Removing non-stationary and irrelevant features,339,0.005,1953800,-3908000,-3907000,1.326,2.0,5.0,5.5,7.0,6.0,7.0,6.0
Model_6,Removing Lag Features,470,0.004,1953500,-3907000,-3906000,1.324,5.0,7.0,8.0,8.5,7.0,9.0,8.0
Model_7,"Removing lag,non-stationary,irrelevant features",149,0.003,1953000,-3906000,-3906000,1.323,1.0,8.5,8.0,10.0,9.0,10.0,9.0
Model_8,Converting non-stationary to stationary features,2190,0.009,1954100,-3908000,-3907000,1.333,10.0,5.0,5.5,1.5,5.0,1.0,5.0
Model_9,Converting non-stationary to stationary and re...,503,0.006,1953200,-3906000,-3906000,1.328,6.0,8.5,8.0,5.5,8.0,5.0,7.0


### Note: Here we will be training Top 5 Models

## Top 5 Models

In [17]:
top5_models=model_comparison[model_comparison.Model_overall_rank<=5.0].sort_values("Model_overall_rank")

In [18]:
top5_models

,Model Description,Cond_No,R-Squared,Log-Likelihood,AIC,BIC,Durbin-Watson,Cond_No_rank,AIC_rank,BIC_rank,R-Squared_rank,Log-Likelihood_rank,Durbin-Watson_rank,Model_overall_rank
Models,,,,,,,,,,,,,,
Model_1,Using all features,841,0.009,1955100,-3910000,-3909000,1.332,9.0,1.0,1.5,1.5,1.0,2.0,1.0
Model_4,Removing non-stationary features,835,0.008,1954700,-3909000,-3909000,1.331,8.0,2.5,1.5,3.0,2.5,3.0,2.0
Model_2,Removing common irrelevant features,731,0.007,1954700,-3909000,-3908000,1.330,7.0,2.5,3.5,4.0,2.5,4.0,3.0
Model_3,Removing irrelevant features,357,0.006,1954200,-3908000,-3908000,1.327,3.0,5.0,3.5,5.5,4.0,6.0,4.0
Model_8,Converting non-stationary to stationary features,2190,0.009,1954100,-3908000,-3907000,1.333,10.0,5.0,5.5,1.5,5.0,1.0,5.0


# Splitting Data into Training and Testing

In [19]:
data_start_date=data.index.get_level_values("date")[0]
data_end_date=data.index.get_level_values("date")[-1]

In [20]:
print(f"Data Start Date: {data_start_date}")
print(f"Data End Date: {data_end_date}")

Data Start Date: 2010-02-22 00:00:00
Data End Date: 2017-11-29 00:00:00


## Training Data

In [21]:
train_start_date=pd.to_datetime("2010-02-22")
train_end_date=pd.to_datetime("2016-02-23")

In [22]:
data_train=data[(data.index.get_level_values("date")>=train_start_date) &
                (data.index.get_level_values("date")<=train_end_date)]

In [23]:
data_train

open       high        low      close      volume  \
date       ticker                                                           
2010-02-22 A       21.347311  21.347311  21.047125  21.251797   2888500.0   
           ACGL    24.070000  24.306667  24.070000  24.230000   2048400.0   
           ACN     34.573465  34.657811  34.202342  34.328862   2421400.0   
           ADI     24.188871  24.261559  23.898120  24.140412   4298200.0   
           ADM     24.683991  24.866038  24.535043  24.667441   3619800.0   
...                      ...        ...        ...        ...         ...   
2016-02-23 XEL     37.379842  37.720174  37.209675  37.653998   2925474.0   
           XOM     76.980304  77.337085  75.937187  76.266741  11039235.0   
           YUM     49.548132  50.054291  49.229182  49.582800   3438179.0   
           ZBH     93.429300  94.119153  91.596000  92.216957   1445740.0   
           ZBRA    68.070000  69.329900  67.420000  68.860000    604016.0   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2010-02-22 A        6.138582e+07      8.755057e+07       8.758384e+07   
           ACGL     4.963273e+07      4.717724e+07       4.194648e+07   
           ACN      8.312391e+07      1.116830e+08       1.292910e+08   
           ADI      1.037603e+08      1.565638e+08       1.256450e+08   
           ADM      8.929120e+07      1.197275e+08       1.496057e+08   
...                          ...               ...                ...   
2016-02-23 XEL      1.101558e+08      1.317412e+08       1.718241e+08   
           XOM      8.419265e+08      1.139976e+09       1.411742e+09   
           YUM      1.704745e+08      1.932388e+08       2.733121e+08   
           ZBH      1.333217e+08      1.051946e+08       1.451527e+08   
           ZBRA     4.159254e+07      4.466156e+07       4.584656e+07   

                   dollar_volume_21d  dollar_volume_rank  ...  bb_upper  \
date       ticker                                         ...             
2010-02-22 A            7.986180e+07               197.0  ...  0.003026   
           ACGL         3.907178e+07               268.0  ...  0.008969   
           ACN          1.358379e+08               119.0  ...  0.033302   
           ADI          1.216604e+08               133.0  ... -0.005210   
           ADM          1.499746e+08               108.0  ...  0.035304   
...                              ...                 ...  ...       ...   
2016-02-23 XEL          1.625370e+08               185.0  ...  0.018735   
           XOM          1.386834e+09                 5.0  ...  0.036861   
           YUM          2.518036e+08               114.0  ...  0.039860   
           ZBH          1.533545e+08               195.0  ...  0.078559   
           ZBRA         4.563769e+07               343.0  ...  0.013873   

                   avg_true_range      macd  macd_hist  macd_signal  \
date       ticker                                                     
2010-02-22 A             0.473915  0.056859   0.260081    -0.203223   
           ACGL          0.346561  0.061855   0.111841    -0.049985   
           ACN           0.674139 -0.479265   0.174337    -0.653602   
           ADI           0.650486 -0.198539   0.494893    -0.693432   
           ADM           0.516744 -0.184526   0.023590    -0.208116   
...                           ...       ...        ...          ...   
2016-02-23 XEL           0.754004  0.685395  -0.015958     0.701353   
           XOM           1.902412  1.442273   0.284922     1.157351   
           YUM           1.329131  0.091311   0.272328    -0.181017   
           ZBH           2.385906 -1.532573   0.172325    -1.704898   
           ZBRA          2.892381  1.612533   1.585095     0.027438   

                   forward_returns_1d  forward_returns_5d  forward_returns_7d  \
date       ticker                                                               
2010-02-22

## Testing Data

In [24]:
test_start_date=pd.to_datetime("2016-02-24")
test_end_date=pd.to_datetime("2017-11-29")

In [25]:
data_test=data[(data.index.get_level_values("date")>=test_start_date)&
               (data.index.get_level_values("date")<=test_end_date)]

In [26]:
data_test

open        high         low       close     volume  \
date       ticker                                                              
2016-02-24 A        36.012987   36.848213   35.836115   36.828561  1454952.0   
           ACGL     67.690000   68.950000   67.300000   68.790000   341478.0   
           ACN      94.798058   95.730182   94.096562   95.662916  2567838.0   
           ADI      48.808218   50.013954   48.325923   49.975371  2690936.0   
           ADM      31.839203   32.180846   31.393170   32.123906  3769583.0   
...                       ...         ...         ...         ...        ...   
2017-11-29 XEL      51.180000   51.480000   50.850000   51.260000  2311537.0   
           XOM      81.650000   82.310000   81.480000   82.270000  9475070.0   
           YUM      81.290000   82.120000   81.200000   81.810000  1658544.0   
           ZBH     115.500000  116.740000  115.000000  116.520000  1749712.0   
           ZBRA    111.630000  112.390000  108.575000  109.430000   318334.0   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2016-02-24 A        5.358379e+07      9.158753e+07       1.064257e+08   
           ACGL     2.349027e+07      2.759548e+07       3.156245e+07   
           ACN      2.456469e+08      2.791125e+08       3.629054e+08   
           ADI      1.344805e+08      1.387501e+08       1.309856e+08   
           ADM      1.210937e+08      1.281672e+08       1.582446e+08   
...                          ...               ...                ...   
2017-11-29 XEL      1.184894e+08      1.072401e+08       1.381043e+08   
           XOM      7.795140e+08      6.577386e+08       7.044767e+08   
           YUM      1.356855e+08      1.058911e+08       1.232016e+08   
           ZBH      2.038764e+08      1.181175e+08       1.283747e+08   
           ZBRA     3.483529e+07      2.512778e+07       3.858277e+07   

                   dollar_volume_21d  dollar_volume_rank  ...  bb_upper  \
date       ticker                                         ...             
2016-02-24 A            1.004817e+08               272.0  ...  0.029179   
           ACGL         3.263230e+07               357.0  ...  0.012149   
           ACN          3.588292e+08                70.0  ...  0.067704   
           ADI          1.330788e+08               226.0  ...  0.035086   
           ADM          1.753127e+08               172.0  ...  0.036638   
...                              ...                 ...  ...       ...   
2017-11-29 XEL          1.471875e+08               187.0  ...  0.012771   
           XOM          6.980541e+08                19.0  ...  0.028160   
           YUM          1.427005e+08               192.0  ...  0.009228   
           ZBH          1.856724e+08               151.0  ...  0.015251   
           ZBRA         3.879641e+07               357.0  ...  0.058111   

                   avg_true_range      macd  macd_hist  macd_signal  \
date       ticker                                                     
2016-02-24 A             1.138302 -0.190118   0.251942    -0.442061   
           ACGL          1.851974  0.193979   0.237189    -0.043210   
           ACN           2.260233 -0.738708   0.414593    -1.153301   
           ADI           1.491596 -0.116798   0.334717    -0.451515   
           ADM           1.108522 -0.079348   0.150722    -0.230070   
...                           ...       ...        ...          ...   
2017-11-29 XEL           0.538257  0.529041  -0.011455     0.540496   
           XOM           0.746925 -0.283034  -0.052811    -0.230222   
           YUM           1.033082  1.068747   0.145776     0.922972   
           ZBH           2.028270 -0.638798   0.623590    -1.262388   
           ZBRA          2.555431 -0.258970   0.158463    -0.417433   

                   forward_returns_1d  forward_returns_5d  forward_returns_7d  \
date       ticker                                    

### Saving Test Data

In [93]:
data_test.to_csv(
    os.path.join(data_store,"test_data.csv"),
    index=True
)

# Preparing Training Data

## Separating Features and Target Variables

In [27]:
train_Y=data_train.filter(like="forward_returns")

In [28]:
train_X=data_train.drop(train_Y.columns,axis=1,inplace=False)

## Data Standardization

In [29]:
train_X_standarized=(train_X
         .groupby("ticker")
         .transform(lambda x: (x-x.mean())/x.std())
        )

### Saving Each Ticker Feature's Mean and Std Deviation

#### Note: This can be used for standardizing the Testing Data

#### Saving Feature's Mean Values

In [30]:
feature_mean_per_ticker=(train_X
                         .groupby("ticker")
                         .apply(lambda x: x.mean())
                        )

In [31]:
feature_mean_per_ticker.to_csv(
    os.path.join(data_store,"training_feature_mean_per_ticker.csv"),
    index=True
)

#### Saving Feature's Standard Deviation Values

In [32]:
feature_std_dev_per_ticker=(train_X
                            .groupby("ticker")
                            .apply(lambda x: x.std())
                           )

In [33]:
feature_std_dev_per_ticker.to_csv(
    os.path.join(data_store,"training_features_std_dev_per_ticker.csv"),
    index=True
)

In [34]:
train_X=train_X_standarized

In [35]:
train_X

open      high       low     close    volume  \
date       ticker                                                     
2010-02-22 A      -1.571355 -1.621298 -1.566455 -1.588083 -0.156541   
           ACGL   -1.576142 -1.574122 -1.564216 -1.568240  3.054903   
           ACN    -1.503369 -1.518475 -1.506436 -1.519097 -0.338131   
           ADI    -1.462160 -1.479389 -1.463275 -1.469124  0.904611   
           ADM    -0.916182 -0.923291 -0.902353 -0.917975 -0.511968   
...                     ...       ...       ...       ...       ...   
2016-02-23 XEL     2.407518  2.411885  2.433889  2.451646  0.019789   
           XOM     0.441488  0.424918  0.396251  0.373889 -0.689229   
           YUM     0.704938  0.711610  0.717225  0.707216 -0.155420   
           ZBH     0.796296  0.791065  0.747731  0.740059 -0.062927   
           ZBRA    0.685893  0.706261  0.695372  0.720551  0.941935   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2010-02-22 A           -0.693067         -0.279116          -0.341175   
           ACGL         1.343749          1.911222           1.708494   
           ACN         -0.758531         -0.957332          -0.950981   
           ADI         -0.044209          1.348163           0.612157   
           ADM         -0.916221         -0.615638           0.118269   
...                          ...               ...                ...   
2016-02-23 XEL          0.918381          1.835798           3.370234   
           XOM         -0.714623         -0.035235           0.951681   
           YUM          0.092064          0.458539           1.878205   
           ZBH          0.442415         -0.049787           1.221948   
           ZBRA         0.907915          1.284902           1.452440   

                   dollar_volume_21d  dollar_volume_rank  ...  max_price_7d  \
date       ticker                                         ...                 
2010-02-22 A               -0.708899            0.014026  ...     -1.698506   
           ACGL             1.505118           -2.431023  ...     -1.574045   
           ACN             -0.963062            1.277944  ...     -1.529100   
           ADI              0.524131           -1.456467  ...     -1.513738   
           ADM              0.143165           -0.951449  ...     -0.967399   
...                              ...                 ...  ...           ...   
2016-02-23 XEL              3.235384           -1.767585  ...      2.360262   
           XOM              0.907162           -0.382588  ...      0.385850   
           YUM              1.685877           -0.314443  ...      0.643528   
           ZBH              1.633233            0.406150  ...      0.756699   
           ZBRA             1.484118           -0.243752  ...      0.644209   

                   max_price_15d  max_price_21d       rsi  bb_lower  bb_upper  \
date       ticker                                                               
2010-02-22 A           -1.765908      -1.805616  1.054291  0.995363 -0.970205   
           ACGL        -1.577768      -1.577905  0.378266  0.452331 -0.631467   
           ACN         -1.536885      -1.526227 -1.015514 -0.732887 -0.034553   
           ADI         -1.533649      -1.541487  0.919647  2.052763 -1.320108   
           ADM         -0.862954      -0.888895 -0.538155 -0.582187 -0.188531   
...                          ...            ...       ...       ...       ...   
2016-02-23 XEL          2.308206       2.281543  1.256485  1.384522 -0.184640   
           XOM          0.329605       0.297865  0.508361  1.772654  0.191105   
           YUM          0.618603       0.631358  0.117181  0.929545  0.083859   
           ZBH          0.837428       0.981501 -1.029809 -0.074962  1.317450   
           ZBRA         0.592091       0.563529  0.891800  3.425449 -0.673004   

                   avg_true_range      macd  macd_hist  macd_signal

## Saving Training's Target Data

In [94]:
train_Y

forward_returns_1d  forward_returns_5d  forward_returns_7d  \
date       ticker                                                               
2010-02-22 A                -0.009631            0.020025            0.002472   
           ACGL             -0.000275            0.013788           -0.005998   
           ACN              -0.010811            0.003002            0.000985   
           ADI              -0.024757            0.019836           -0.007765   
           ADM              -0.006374            0.010218           -0.002345   
...                               ...                 ...                 ...   
2016-02-23 XEL               0.002009           -0.002023            0.000253   
           XOM               0.003570            0.014099           -0.003628   
           YUM              -0.000420            0.051332            0.012996   
           ZBH               0.008444            0.005991            0.014881   
           ZBRA              0.017136            0.017967            0.015730   

                   forward_returns_15d  forward_returns_21d  
date       ticker                                            
2010-02-22 A                 -0.003291             0.011566  
           ACGL              -0.002398             0.001330  
           ACN                0.009022            -0.000707  
           ADI               -0.002070             0.011972  
           ADM               -0.001745            -0.000341  
...                                ...                  ...  
2016-02-23 XEL                0.001979             0.008585  
           XOM                0.004975            -0.004398  
           YUM                0.008769             0.019878  
           ZBH               -0.007044             0.000192  
           ZBRA              -0.036609            -0.018077  

[565488 rows x 5 columns]

In [95]:
train_Y.to_csv(
    os.path.join(data_store,"train_Y.csv"),
    index=True
)

## Preparing Model's Data

### Model-1: Using All Features

In [36]:
model1_train_X=train_X.copy()

#### Saving Model1 Train_X Data

In [96]:
model1_train_X.to_csv(
    os.path.join(data_store,"model1_train_X.csv"),
    index=True
)

### Model-4: Removing Non-stationary Features

In [37]:
non_stationary_features

['open',
 'high',
 'low',
 'close',
 'sma_7d',
 'sma_15d',
 'sma_21d',
 'min_price_7d',
 'min_price_15d',
 'min_price_21d',
 'max_price_7d',
 'max_price_15d',
 'max_price_21d',
 'avg_true_range']

In [38]:
model4_train_X=train_X.drop(non_stationary_features,axis=1,inplace=False)


#### Saving Model4 Train_X Data

In [97]:
model4_train_X.to_csv(
    os.path.join(data_store,"model4_train_X.csv"),
    index=True
)

### Model-2: Removing Common Irrelevant Features

In [39]:
common_irrelevant_features

['returns_15d_lag5', 'returns_21d_lag3', 'momentum_21d_lag3']

In [40]:
model2_train_X=train_X.drop(common_irrelevant_features,axis=1,inplace=False)

#### Saving Model2 Train_X Data

In [98]:
model2_train_X.to_csv(
    os.path.join(data_store,"model2_train_X.csv"),
    index=True
)

### Model-3: Removing Irrelevant Features

In [41]:
irrelevant_features

['dollar_volume',
 'dollar_volume_7d',
 'dollar_volume_15d',
 'returns_7d',
 'returns_15d',
 'returns_1d_lag5',
 'returns_7d_lag5',
 'returns_7d_lag7',
 'returns_15d_lag1',
 'returns_15d_lag3',
 'returns_15d_lag5',
 'returns_15d_lag7',
 'returns_21d_lag3',
 'returns_21d_lag7',
 'momentum_15d',
 'momentum_7d_lag5',
 'momentum_7d_lag7',
 'momentum_15d_lag1',
 'momentum_15d_lag3',
 'momentum_15d_lag5',
 'momentum_21d_lag3',
 'momentum_21d_lag5',
 'momentum_21d_lag7',
 'volatility_7d',
 'volatility_7d_lag1',
 'volatility_7d_lag3',
 'volatility_15d_lag3',
 'volatility_21d_lag7',
 'sma_15d',
 'sma_21d',
 'min_price_21d',
 'max_price_15d',
 'max_price_21d']

In [42]:
model3_train_X=train_X.drop(irrelevant_features,axis=1,inplace=False)


#### Saving Model3 Train_X Data

In [99]:
model3_train_X.to_csv(
    os.path.join(data_store,"model3_train_X.csv"),
    index=True
)

### Model-8: Converting Non Stationary Features to Stationary

In [43]:
non_stationary_features

['open',
 'high',
 'low',
 'close',
 'sma_7d',
 'sma_15d',
 'sma_21d',
 'min_price_7d',
 'min_price_15d',
 'min_price_21d',
 'max_price_7d',
 'max_price_15d',
 'max_price_21d',
 'avg_true_range']

In [44]:
model8_train_X=train_X.copy()

In [45]:
for feat in non_stationary_features:
    model8_train_X[feat]=(model8_train_X
                          .groupby("ticker")[feat]
                          .transform(lambda x: x.diff(periods=1))
                         )
    

In [46]:
model8_train_X.dropna(inplace=True)

#### Saving Model8 Train_X Data

In [100]:
model8_train_X.to_csv(
    os.path.join(data_store,"model8_train_X.csv"),
    index=True
)

# Model Training

In [47]:
training_period=63
test_period=10
n_splits=int(6*252/test_period)
lookahead=1
target="forward_returns_1d"

## Model-1 Training

In [48]:
model1_lr=LinearRegression()

In [49]:
model1_trainer=TrainModel(
    model=model1_lr,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target=target
)

In [50]:
trained_model1,model1_predictions,model1_scores=model1_trainer.train(X=model1_train_X,Y=train_Y)

### Saving Model

In [54]:
joblib.dump(
    trained_model1,
    os.path.join(data_store,"model1.pkl")
)

['Prepared_Data_Store/model1.pkl']

### Saving Model Predictions

In [55]:
model1_predictions

yreal    ypreds
date       ticker                    
2016-02-09 A       0.013571 -0.005870
           ACGL    0.014881 -0.000842
           ACN     0.009537  0.004813
           ADI    -0.007098 -0.002929
           ADM    -0.011912 -0.005405
...                     ...       ...
2010-06-17 XEL     0.005126 -0.012106
           XOM     0.007987 -0.021737
           YUM    -0.002114 -0.002609
           ZBH     0.000727 -0.004634
           ZBRA    0.004384 -0.004532

[538560 rows x 2 columns]

In [56]:
model1_predictions.to_csv(
    os.path.join(data_store,"model1_predictions.csv"),
    index=True
    
)

### Saving Model Scores

In [57]:
model1_scores

,ic,rmse
date,,
2016-02-09,6.350212,0.022171
2016-02-10,9.778911,0.025307
2016-02-11,-26.910616,0.037713
2016-02-12,-13.418210,0.027853
2016-02-16,-9.974968,0.031515
...,...,...
2010-06-11,22.169961,0.013511
2010-06-14,9.917208,0.029917
2010-06-15,18.155798,0.017353


In [58]:
model1_scores.to_csv(
    os.path.join(data_store,"model1_scores.csv"),
    index=True
)

## Model-4 Training

In [59]:
model4_lr=LinearRegression()

In [60]:
model4_trainer=TrainModel(
    model=model4_lr,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target=target
)

In [61]:
trained_model4,model4_predictions,model4_scores=model4_trainer.train(X=model4_train_X,Y=train_Y)

### Saving Model

In [62]:
joblib.dump(
    trained_model4,
    os.path.join(data_store,"model4.pkl")
)

['Prepared_Data_Store/model4.pkl']

### Saving Model Predictions

In [63]:
model4_predictions.to_csv(
    os.path.join(data_store,"model4_predictions.csv"),
    index=True
    
)

### Saving Model Scores

In [64]:
model4_scores.to_csv(
    os.path.join(data_store,"model4_scores.csv"),
    index=True
)

## Model-2 Training

In [65]:
model2_lr=LinearRegression()

In [66]:
model2_trainer=TrainModel(
    model=model2_lr,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target=target
)

In [67]:
trained_model2,model2_predictions,model2_scores=model2_trainer.train(X=model2_train_X,Y=train_Y)

### Saving Model

In [68]:
joblib.dump(
    trained_model2,
    os.path.join(data_store,"model2.pkl")
)

['Prepared_Data_Store/model2.pkl']

### Saving Model Predictions

In [69]:
model2_predictions.to_csv(
    os.path.join(data_store,"model2_predictions.csv"),
    index=True
    
)

### Saving Model Scores

In [70]:
model2_scores.to_csv(
    os.path.join(data_store,"model2_scores.csv"),
    index=True
)

## Model-3 Training

In [71]:
model3_lr=LinearRegression()

In [72]:
model3_trainer=TrainModel(
    model=model3_lr,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target=target
)

In [73]:
trained_model3,model3_predictions,model3_scores=model3_trainer.train(X=model3_train_X,Y=train_Y)

### Saving Model

In [74]:
joblib.dump(
    trained_model3,
    os.path.join(data_store,"model3.pkl")
)

['Prepared_Data_Store/model3.pkl']

### Saving Model Predictions

In [75]:
model3_predictions.to_csv(
    os.path.join(data_store,"model3_predictions.csv"),
    index=True
    
)

### Saving Model Scores

In [76]:
model3_scores.to_csv(
    os.path.join(data_store,"model3_scores.csv"),
    index=True
)

## Model-8 Training

In [86]:
idx=pd.IndexSlice
model8_trainY=train_Y.loc[idx[pd.to_datetime("2010-02-23"):,:]]

In [87]:
model8_lr=LinearRegression()

In [88]:
model8_trainer=TrainModel(
    model=model8_lr,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target=target
)

In [89]:
trained_model8,model8_predictions,model8_scores=model8_trainer.train(X=model8_train_X,Y=model8_trainY)

### Saving Model

In [90]:
joblib.dump(
    trained_model8,
    os.path.join(data_store,"model8.pkl")
)

['Prepared_Data_Store/model8.pkl']

### Saving Model Predictions

In [91]:
model8_predictions.to_csv(
    os.path.join(data_store,"model8_predictions.csv"),
    index=True
    
)

### Saving Model Scores

In [92]:
model8_scores.to_csv(
    os.path.join(data_store,"model8_scores.csv"),
    index=True
)

# END